In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# Run store_key notebook to ensure secrets are configured
dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=60
)

# Load exchangerate API key from secrets (for reference)
exchange_key = dbutils.secrets.get(
    scope="fin-project",
    key="exchangerate_key"
)

print(f"✅ API key loaded: {exchange_key[:4]}****")
print("ℹ️ Using Frankfurter API (free, no auth required)\n")

# Define major currency pairs to track
base_currency = "USD"
target_currencies = ["EUR", "GBP", "JPY", "CHF", "CAD", "AUD", "NZD", "CNY", "INR", "SGD"]

# Fetch forex data from Frankfurter API (free, open-source, no API key needed)
api_url = f"https://api.frankfurter.app/latest?from={base_currency}"

print(f"🔄 Fetching forex exchange rates from Frankfurter API...")
response = requests.get(api_url)

if response.status_code != 200:
    raise Exception(f"❌ API request failed: {response.status_code} - {response.text}")

data = response.json()

if "rates" not in data:
    raise Exception(f"❌ No rates data returned from API")

# Extract conversion rates
rates = data.get("rates", {})
base = data.get("base", base_currency)
date_str = data.get("date", "")
timestamp = datetime.now()

print(f"Base Currency: {base}")
print(f"Rate Date: {date_str}\n")

# Create list of forex data
forex_data = []

for currency in target_currencies:
    if currency in rates:
        rate = rates[currency]
        forex_data.append(Row(
            base_currency=base,
            target_currency=currency,
            exchange_rate=rate,
            pair=f"{base}/{currency}",
            rate_date=date_str,
            fetch_timestamp=timestamp,
            source="frankfurter"
        ))
        print(f"  {base}/{currency}: {rate:.4f}")
    else:
        print(f"  ⚠️ {currency} not available")

if not forex_data:
    raise Exception("❌ No forex data found in API response")

# Create DataFrame
schema = StructType([
    StructField("base_currency", StringType(), False),
    StructField("target_currency", StringType(), False),
    StructField("exchange_rate", DoubleType(), True),
    StructField("pair", StringType(), True),
    StructField("rate_date", StringType(), True),
    StructField("fetch_timestamp", TimestampType(), False),
    StructField("source", StringType(), True)
])

df = spark.createDataFrame(forex_data, schema=schema)

print(f"\n✅ Fetched {df.count()} forex pairs")

# Create database and schema if not exists
spark.sql("CREATE DATABASE IF NOT EXISTS financial_market")
spark.sql("CREATE SCHEMA IF NOT EXISTS financial_market.bronze")

# Write to bronze table
table_name = "financial_market.bronze.forex"

df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(table_name)

print(f"✅ Data stored in {table_name}")
print(f"✅ Run timestamp: {timestamp}")

# Display sample
display(df)